In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PINN(nn.Module):
    def __init__(self, layers=[2, 64, 64, 64, 64, 1]):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers)-2):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))
            self.layers.append(nn.Tanh())
        self.layers.append(nn.Linear(layers[-2], layers[-1]))
    
    def forward(self, X):
        for layer in self.layers:
            X = layer(X)
        return X

# Parameters
nu = 0.01 / np.pi
N_colloc = 20000
# Collocation points (interior)
colloc_x = 2 * torch.rand(N_colloc, 1, requires_grad=True, device=device) - 1  # x in [-1,1]
colloc_t = torch.rand(N_colloc, 1, requires_grad=True, device=device)          # t in [0,1]
colloc = torch.cat([colloc_x, colloc_t], dim=1)
colloc[:, 0] = 2 * colloc[:, 0] - 1  # x in [-1,1]
colloc[:, 1] = colloc[:, 1]          # t in [0,1]

# Initial condition points
ic_x = 2 * torch.rand(N_ic, 1, device=device) - 1
ic_t = torch.zeros_like(ic_x)
ic = torch.cat([ic_x, ic_t], dim=1)
ic.requires_grad = True
u_ic_true = -torch.sin(np.pi * ic_x)

model = PINN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 15000
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # PDE residual
    u = model(colloc)
    grads = torch.autograd.grad(u, colloc, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_t = grads[:, 1:2]
    u_x = grads[:, 0:1]
    u_xx = torch.autograd.grad(u_x, colloc, grad_outputs=torch.ones_like(u_x), create_graph=True)[0][:, 0:1]
    
    residual = u_t + u * u_x - nu * u_xx
    loss_pde = torch.mean(residual**2)
    
    # Initial condition
    u_ic = model(ic)
    loss_ic = torch.mean((u_ic - u_ic_true)**2)
    
    # Periodic BCs approximated softly (or enforce exactly via network design—advanced)
    loss_bc = 0  # For simplicity here; in practice add points on boundaries
    
    loss = loss_pde + 10 * loss_ic  # Weight IC more if needed
    loss.backward(retain_graph=True)
    optimizer.step()
    
    if epoch % 2000 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}, PDE: {loss_pde.item():.6f}, IC: {loss_ic.item():.6f}")

# Visualisation
with torch.no_grad():
    x = torch.linspace(-1, 1, 400)
    t = torch.linspace(0, 1, 200)
    X, T = torch.meshgrid(x, t, indexing='ij')
    XT = torch.stack([X.flatten(), T.flatten()], dim=1).to(device)
    u_pred = model(XT).cpu().numpy().reshape(400, 200)
    
    plt.figure(figsize=(10, 6))
    plt.contourf(T.numpy(), X.numpy(), u_pred.T, levels=50, cmap='viridis')
    plt.colorbar(label='u(x,t)')
    plt.xlabel('t')
    plt.ylabel('x')
    plt.title('PINN Solution to Burgers\' Equation')
    plt.show()

NameError: name 'N_ic' is not defined

# Session 8: Implementation in Python (Part 2) – Inverse problems, loss balancing, and 2D Poisson Equation

We are building on the forward Burgers' equation from Session 7. In this session we tackle **inverse problems** (inferring unknown parameters from noisy data), discuss the crucial art of **loss weighting**, and implement a **2D elliptic PDE**: the Poisson equation. Inverse problems are where PINNs truly excel in real-world physics—think identifying material properties from experiments or damping coefficients from oscillations.

## 1. Inverse Problems in PINNs

In inverse settings, unknown parameters (e.g., viscosity $\nu$) become learnable alongside the solution field.

Typical architectures for inverse PINNs often include the parameters as trainable scalars:



![](figures/fig8_1.png)
![](figures/fig8_2.png)
![](figures/fig8_3.png)






Parameter identification diagrams:



![](figures/fig8_4.png)
![](figures/fig8_5.png)
![](figures/fig8_6.png)








#### 2. Hands-On: Inverse Burgers' Equation with Noisy Data
We'll modify last week's code: add sparse noisy measurements and make $\nu$ a learnable parameter.

```python
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class InversePINN(nn.Module):
    def __init__(self, layers=[2, 64, 64, 64, 64, 1]):
        super().__init__()
        self.net = nn.Sequential(
            *[item for _ in range(4) for item in (nn.Linear(layers[_], layers[_+1]), nn.Tanh())],
            nn.Linear(layers[-2], layers[-1])
        )
        self.nu = nn.Parameter(torch.tensor(0.01/np.pi, device=device))  # initial guess
    
    def forward(self, X):
        return self.net(X)

# True nu and generate "observed" noisy data
true_nu = 0.01 / np.pi
N_data = 1000  # sparse noisy points
data_x = 2 * torch.rand(N_data, 1) - 1
data_t = torch.rand(N_data, 1)
data_xt = torch.cat([data_x, data_t], dim=1).to(device)

# Simulate true solution (simple forward model - in practice this would be real data)
# Here we use a quick analytical approximation for demo
u_true = -torch.sin(np.pi * data_x) * torch.exp(-true_nu * np.pi**2 * data_t)
noise = 0.01 * torch.randn_like(u_true)
u_noisy = u_true + noise
u_noisy = u_noisy.to(device)

# Collocation and IC as before
N_colloc = 20000
colloc = torch.rand(N_colloc, 2, requires_grad=True, device=device)
colloc[:, 0] = 2 * colloc[:, 0] - 1
colloc[:, 1] = colloc[:, 1]

N_ic = 500
ic_x = 2 * torch.rand(N_ic, 1, device=device) - 1
ic_t = torch.zeros_like(ic_x)
ic = torch.cat([ic_x, ic_t], dim=1)
ic.requires_grad = True
u_ic_true = -torch.sin(np.pi * ic_x)

model = InversePINN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 20000
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # PDE residual
    u = model(colloc)
    grads = torch.autograd.grad(u, colloc, torch.ones_like(u), create_graph=True)[0]
    u_t, u_x = grads[:, 1:2], grads[:, 0:1]
    u_xx = torch.autograd.grad(u_x, colloc, torch.ones_like(u_x), create_graph=True)[0][:, 0:1]
    residual = u_t + u * u_x - self.nu * u_xx
    loss_pde = torch.mean(residual**2)
    
    # IC
    u_ic = model(ic)
    loss_ic = torch.mean((u_ic - u_ic_true)**2)
    
    # Data loss (noisy measurements)
    u_data = model(data_xt)
    loss_data = torch.mean((u_data - u_noisy)**2)
    
    # Total loss - note weighting!
    loss = loss_pde + 10 * loss_ic + 50 * loss_data  # data often needs higher weight
    
    loss.backward()
    optimizer.step()
    
    if epoch % 2000 == 0:
        print(f"Epoch {epoch}, Total: {loss.item():.6f}, PDE: {loss_pde.item():.6f}, "
              f"IC: {loss_ic.item():.6f}, Data: {loss_data.item():.6f}, nu: {model.nu.item():.6f}")

print(f"Inferred nu: {model.nu.item():.6f} (true: {true_nu:.6f})")
```

Typical results with noisy data:








#### 3. Loss Weighting Strategies
Multi-term losses often compete—bad weighting leads to poor convergence. Common approaches:












- Manual tuning (as above).
- Adaptive weights (e.g., NTK-based or gradient balancing).
- Start with equal weights, then adjust.

#### 4. 2D Poisson Equation
Steady-state: $\nabla^2 u = f(x,y)$ in $[0,1]^2$, with $u=0$ on boundaries.

Example $f = -2\pi^2 \sin(\pi x) \sin(\pi y)$, exact $u = \sin(\pi x) \sin(\pi y)$.

Quick implementation sketch (full code similar to above, input dim=2, compute Laplacian).

Typical PINN solutions:
















**Assignment**: Implement the inverse Burgers' code, report inferred $\nu$ with different noise levels. Bonus: Add adaptive weighting.

Next weeks: Advanced topics—stiff problems, conservative PINNs, and project ideas.

You're now equipped for real scientific discovery with PINNs! Questions?

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class PINN(nn.Module):
    def __init__(self, layers=[2, 64, 64, 64, 64, 1]):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers)-2):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))
            self.layers.append(nn.Tanh())
        self.layers.append(nn.Linear(layers[-2], layers[-1]))
    
    def forward(self, X):
        for layer in self.layers:
            X = layer(X)
        return X

# Parameters
nu = 0.01 / np.pi
N_colloc = 20000
# Collocation points (interior)
colloc_x = 2 * torch.rand(N_colloc, 1, requires_grad=True, device=device) - 1  # x in [-1,1]
colloc_t = torch.rand(N_colloc, 1, requires_grad=True, device=device)          # t in [0,1]
colloc = torch.cat([colloc_x, colloc_t], dim=1)
colloc[:, 0] = 2 * colloc[:, 0] - 1  # x in [-1,1]
colloc[:, 1] = colloc[:, 1]          # t in [0,1]

# Initial condition points
ic_x = 2 * torch.rand(N_ic, 1, device=device) - 1
ic_t = torch.zeros_like(ic_x)
ic = torch.cat([ic_x, ic_t], dim=1)
ic.requires_grad = True
u_ic_true = -torch.sin(np.pi * ic_x)

model = PINN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 15000
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # PDE residual
    u = model(colloc)
    grads = torch.autograd.grad(u, colloc, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_t = grads[:, 1:2]
    u_x = grads[:, 0:1]
    u_xx = torch.autograd.grad(u_x, colloc, grad_outputs=torch.ones_like(u_x), create_graph=True)[0][:, 0:1]
    
    residual = u_t + u * u_x - nu * u_xx
    loss_pde = torch.mean(residual**2)
    
    # Initial condition
    u_ic = model(ic)
    loss_ic = torch.mean((u_ic - u_ic_true)**2)
    
    # Periodic BCs approximated softly (or enforce exactly via network design—advanced)
    loss_bc = 0  # For simplicity here; in practice add points on boundaries
    
    loss = loss_pde + 10 * loss_ic  # Weight IC more if needed
    loss.backward(retain_graph=True)
    optimizer.step()
    
    if epoch % 2000 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}, PDE: {loss_pde.item():.6f}, IC: {loss_ic.item():.6f}")

# Visualisation
with torch.no_grad():
    x = torch.linspace(-1, 1, 400)
    t = torch.linspace(0, 1, 200)
    X, T = torch.meshgrid(x, t, indexing='ij')
    XT = torch.stack([X.flatten(), T.flatten()], dim=1).to(device)
    u_pred = model(XT).cpu().numpy().reshape(400, 200)
    
    plt.figure(figsize=(10, 6))
    plt.contourf(T.numpy(), X.numpy(), u_pred.T, levels=50, cmap='viridis')
    plt.colorbar(label='u(x,t)')
    plt.xlabel('t')
    plt.ylabel('x')
    plt.title('PINN Solution to Burgers\' Equation')
    plt.show()